In [ ]:
!pip install autoawq datasets transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for autoawq: filename=autoawq-0.2.9-py3-none-any.whl size=115106 sha256=8af7ba37a1a7a9e797605a1e111616d7b0cfde3661d701532cf86e42dcc6e275
  Stored in directory: /root/.cache/pip/wheels/45/1a/7b/7314b3a958454e8ce349f600829a3f0a6a05aeebf987be1e16
Successfully built autoawq


In [ ]:
import torch
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
from datasets import load_dataset


/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

In [ ]:

# 1. Configuration
model_path = "mistralai/Mistral-7B-v0.3"
quant_path = "Mistral-7B-v0.3-AWQ-4bit-Wikitext"
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM" # GEMM is better for general use/vLLM
}

In [ ]:
# 2. Load Model & Tokenizer to GPU
# device_map="cuda:0" forces the model onto your first GPU
print("📦 Loading model to GPU...")
model = AutoAWQForCausalLM.from_pretrained(
    model_path,
    low_cpu_mem_usage=True,
    use_cache = False
    #device_map="auto",
    #torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)


📦 Loading model to GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:

# 3. Load & Format WikiText-2 Calibration Data
traindata = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
# Filter for meaningful text blocks
calib_data = [text for text in traindata['text'] if len(text) > 128][:128]


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [ ]:
# 4. Execute Quantization
# AutoAWQ uses the GPU kernels to calculate scales based on calib_data
print("🚀 Starting GPU-accelerated quantization...")
model.quantize(
    tokenizer,
    quant_config=quant_config,
    calib_data=calib_data,
    n_parallel_calib_samples = 32
)

🚀 Starting GPU-accelerated quantization...


AWQ: 100%|██████████| 32/32 [1:49:14<00:00, 204.83s/it]


In [ ]:

# 5. Save the Quantized Model
model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)

print(f"✅ GPU Quantization Complete! Saved to: {quant_path}")

Writing model shards: 0it [00:00, ?it/s]

✅ GPU Quantization Complete! Saved to: Mistral-7B-v0.3-AWQ-4bit-Wikitext


In [ ]:
drive_model_folder = "Mistral-7B-v0.3-AWQ-4bit-Wikitext"
model_path = f"/content/drive/My Drive/{drive_model_folder}"

# 3. Verify the path exists and list files
if os.path.exists(model_path):
    print(f"✅ Model found at: {model_path}")
    print("Files in directory:", os.listdir(model_path))
else:
    print(f"❌ Error: Could not find the folder at {model_path}")
    print("Please check your Google Drive folder name and structure.")

✅ Model found at: /content/drive/My Drive/Mistral-7B-v0.3-AWQ-4bit-Wikitext
Files in directory: ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']
